# Configuraci?n Inicial
M?todos un utils en com?n en los notebooks

In [ ]:
# Imports 
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql import functions as f, SparkSession
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import when, col


In [ ]:
class MySQLConnector:
    def __init__(self, spark: SparkSession, connection_properties: dict, url: str):
        self.spark = spark
        self.properties = connection_properties
        self.url = url

    def get_dataframe(self, sql_query: str):        
        df = self.spark.read.jdbc(
            url=self.url,
            table=sql_query,
            properties=self.properties
        )
        return df
    
    def save_db(self, df, tabla):
        df.write.jdbc(
            url=self.url,
            table=tabla,
            mode='append',
            properties=self.properties
        )
        
def create_spark_session(path_jar_driver):    
    conf = SparkConf().set('spark.driver.extraClassPath', path_jar_driver)
    spark_context = SparkContext(conf=conf)
    sql_context = SQLContext(spark_context)
    return sql_context.sparkSession    

def get_dataframe_from_csv(_PATH, _sep):
    return spark.read.load(_PATH, format="csv", sep=_sep, inferSchema="true", header='true')

In [ ]:
db_user = 'Estudiante_65_202415'
db_psswd = 'Estudiante_202010409'

connection_properties = {
    "user": db_user,
    "password": db_psswd,
    "driver": "com.mysql.cj.jdbc.Driver"
}

source_db_string_connection = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional_ETL'
destination_db_string_connection = f'jdbc:mysql://157.253.236.120:8080/{db_user}'

# jdbc:mysql://157.253.236.120:8080/WWImporters_DWH_tablero

# Driver de conexion
# LINUX
path_jar_driver = '/opt/mysql/lib/mysql-connector-java-8.0.28.jar'
# WINDOWS
#path_jar_driver = 'C:\Users\Rodolfo\OneDrive\Maestria MIAD\Semestre 1\Ciclo 2\Modelado de Datos y ETL\Semana 2\mysql-connector-j-9.0.0.jar'
# WINDOWS como esta en la VM 
#path_jar_driver = 'C:\Program Files (x86)\MySQL\Connector J 8.0\mysql-connector-java-8.0.28.jar'

In [ ]:
spark = create_spark_session(path_jar_driver)

In [ ]:
conn_orig = MySQLConnector(spark=spark, connection_properties=connection_properties, url=source_db_string_connection)
conn_dest = MySQLConnector(spark=spark, connection_properties=connection_properties, url=destination_db_string_connection)

# Dimensi?n Condiciones de Pago

![Modelo Movimientos](./images/CondPago.png)

### Extraction

In [ ]:
sql_Rs_CondicionDePago = '''
(
SELECT DISTINCT 
    IdCondicionesDePago_T AS IdCondicionDePago_T, 
    Descripcion, 
    Tipo 
FROM FuenteCondicionesDePago_ETL
ORDER BY IdCondicionDePago_T
) AS CondicionDePago
'''

df_Rs_CondicionDePago = conn_orig.get_dataframe(sql_Rs_CondicionDePago)
df_Rs_CondicionDePago = df_Rs_CondicionDePago \
    .withColumn('IdCondicionDePago_T', col('IdCondicionDePago_T').cast(IntegerType())) \
    .withColumn('Descripcion', col('Descripcion').cast(StringType())) \
    .withColumn('Tipo', col('Tipo').cast(StringType())) \
    .orderBy("IdCondicionDePago_T")


In [ ]:
df_Rs_CondicionDePago.show(40)

## Transformation

Realizar las transformaciones en la variable "Tipo" para que ?nicamente que valores Copago y Coseguro 

In [ ]:
# Realizar las transformaciones en la columna "Tipo"
df_Rs_CondicionDePago = df_Rs_CondicionDePago.withColumn(
    "Tipo",
    when(col("Tipo") == "Copagado", "Copago")
    .when(col("Tipo") == "Coseguridad", "Coseguro")
    .otherwise(col("Tipo"))
)

# Mostrar los resultados para verificar las transformaciones
df_Rs_CondicionDePago.show(40)

Incluir la variable de la base de datos IdCondicionDePago_DWH y organizar la estructura de la tabla

Adicionar comodin cuando la referencia no exista

In [ ]:
df_Rs_CondicionDePago = df_Rs_CondicionDePago.coalesce(1).withColumn('IdCondicionDePago_DWH', f.monotonically_increasing_id() + 1)
df_Rs_CondicionDePago.show(40)

In [ ]:
cond_data = [(0, "Missing", "Missing", 0)]
cond_columns = ["IdCondicionDePago_T", "Descripcion", "Tipo", "IdCondicionDePago_DWH"]
dummy_cond_pago = spark.createDataFrame(cond_data, cond_columns)

df_Rs_CondicionDePago = dummy_cond_pago.union(df_Rs_CondicionDePago)
df_Rs_CondicionDePago.show(40)


## Load

In [ ]:
df_Rs_CondicionDePago.printSchema()

In [ ]:
df_Rs_CondicionDePago = df_Rs_CondicionDePago \
    .withColumn('IdCondicionDePago_DWH', col('IdCondicionDePago_DWH').cast(IntegerType())) \
    .withColumn('IdCondicionDePago_T', col('IdCondicionDePago_T').cast(IntegerType())) \
    .withColumn('Descripcion', col('Descripcion').cast(StringType())) \
    .withColumn('Tipo', col('Tipo').cast(StringType())) 
df_Rs_CondicionDePago.printSchema()


In [ ]:
df_Rs_CondicionDePago.count()

In [ ]:
# Cargue de la dimension
conn_dest.save_db(df_Rs_CondicionDePago, "Rs_CondicionDePago")

# Dimensi?n Proveedor

### Extraction
Aplicando el distinct se garantiza la unicidad

In [ ]:
sql_Rs_Proveedor = "(SELECT DISTINCT IdProveedor_T FROM FuentePlanesBeneficio_ETL) AS Rs_Proveedor"

df_Rs_Proveedor = conn_orig.get_dataframe(sql_Rs_Proveedor)
df_Rs_Proveedor.withColumn('IdProveedor_T', col('IdProveedor_T').cast(IntegerType()))

In [ ]:
df_Rs_Proveedor.show()

In [ ]:
df_Rs_Proveedor.printSchema()

### Seleccionar los valores ?nicos de IdProveedor_T

In [ ]:
df_Rs_Proveedor = df_Rs_Proveedor.select("IdProveedor_T").distinct().orderBy("IdProveedor_T")

df_Rs_Proveedor.show()

In [ ]:
print((df_Rs_Proveedor.count(), len(df_Rs_Proveedor.columns)))

In [ ]:
df_Rs_Proveedor = df_Rs_Proveedor.coalesce(1).withColumn('IdProveedor_DWH', f.monotonically_increasing_id() + 1)
df_Rs_Proveedor = df_Rs_Proveedor.select("IdProveedor_DWH", "IdProveedor_T")
df_Rs_Proveedor.show()

In [ ]:
proveedor_data = [(0, 0)]
proveedor_columns = ["IdProveedor_T", "IdProveedor_DWH"]
dummy_proveedor = spark.createDataFrame(proveedor_data, proveedor_columns)                          

df_Rs_Proveedor = dummy_proveedor.union(df_Rs_Proveedor)
df_Rs_Proveedor.show(40)

## Load

In [ ]:
# CARGUE
conn_dest.save_db(df_Rs_Proveedor, "Rs_Proveedor")

## Dimensi?n NivelesDeServicio

### Extraction

In [ ]:
sql_Rs_NivelesDeServicio = 'RaSaTransaccional_ETL.NivelesDeServicio'

df_Rs_NivelesDeServicio = conn_orig.get_dataframe(sql_Rs_NivelesDeServicio)
df_Rs_NivelesDeServicio = df_Rs_NivelesDeServicio \
    .withColumn('IdNivelDeServicio_DWH', col('IdNivelDeServicio_DWH').cast(IntegerType())) \
    .withColumn('IdNivelDeServicio_T', col('IdNivelDeServicio_T').cast(IntegerType())) \
    .withColumn('Descripcion', col('Descripcion').cast(StringType()))

In [ ]:
df_Rs_NivelesDeServicio.show()

## Transformation

In [ ]:
nivel_data = [(0, 0, "Missing")]
nivel_columns = ["IdNivelDeServicio_DWH", "IdNivelDeServicio_T", "Descripcion"]
dummy_nivel = spark.createDataFrame(nivel_data, nivel_columns)                          

df_Rs_NivelesDeServicio = dummy_nivel.union(df_Rs_NivelesDeServicio)
df_Rs_NivelesDeServicio.show(40)

No existen Transformaciones

## Load

In [ ]:
# CARGUE
conn_dest.save_db(df_Rs_NivelesDeServicio, "Rs_NivelesDeServicio")  